# Binary classification: logistic vs VQC

Classify points by `expect(O, ψ_θ)` per class. Same trace runs classical-bilinear on cpu, parameterized circuit on the simulator route.

**Hardware-agnostic by design.** This notebook traces ONE computation with the uniqx SDK; the engine picks the route (classical / simulator / accelerator) at preflight time.

## Background

**Problem:** binary classification from features. **Classical:** logistic regression. **Quantum:** Variational Quantum Classifier (VQC) — angle-encode features, parameterized ansatz, expect per class.

## Setup

In [ ]:
import os
import numpy as np
from uniqx.core.execution import connect, preflight, submit, get
from uniqx.core.tracing import trace
from uniqx import login

GATEWAY_ADDR = os.environ.get("GATEWAY_ADDR", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=GATEWAY_ADDR)
client = connect(GATEWAY_ADDR)


## Trace the computation

In [ ]:
from uniqx.ops import expect, linalg
from uniqx.ops.primitives.nn import relu

n = 4
rng = np.random.default_rng(10)
W = rng.standard_normal((n, n)) * 0.3
x = np.array([0.4, -0.1, 0.6, -0.3])
M = rng.standard_normal((n, n)) * 0.2
O = 0.5 * (M + M.T)

@trace
def prog(weight, vec, observable):
    h = relu(linalg.matmul(weight, vec))
    return expect(observable, h)

module = prog(W.tolist(), x.tolist(), O.tolist())


## Preflight — see what routes the engine found

uniqx is hardware-agnostic: the same trace runs on every available route. `preflight` returns the engine's choices.

In [ ]:
options = preflight(module, client=client)
print("Available routes:")
for o in options:
    print(f"  {o['label']:25s}  score={o.get('score', 'n/a')}")


## Run on every available route

Production usage is just `client.run(module)` — the engine picks the best route automatically. Here we run on every route for side-by-side comparison.

In [ ]:
runs = {}
for o in options:
    label = o.get("label") or ""
    print(f"\n--- Running on {label} ---")
    job_id = submit(module, client=client, preflight_job_id=options.job_id, option_idx=o['_idx'])
    res = get(job_id, client=client, wait=True, timeout=120)
    runs[label] = res
    print(f"  done: {str(res)[:200]}")
